In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision 
from torchvision.datasets import CIFAR10

In [2]:
#datsets and dataloader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform=transforms.Compose([    
    transforms.ToTensor(),
   transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )])
trainset = CIFAR10(root='./data',train = True , download= False, transform=transform)
testset = CIFAR10(root='./data',train = False , download= False, transform=transform)

In [5]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64)

ARCHITECTURE OF CNN


In [9]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(3,32, kernel_size=3,padding=1),#o/p=input-feature+2padding/stride
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel size=2,stride=2
            nn.Conv2d(32,64, kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64,128, kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )
    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)#flattening
        x=self.fc_layers(x)
        return x
    

In [10]:
model=CNN()

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

TRAINING THE CNN

In [16]:
epochs=10
for epoch in range(epochs):
    epoch_training_loss=0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output,labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()
    print(f"epoch={epoch+1}/{epochs} and loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 and loss=0.12042320632945051
epoch=2/10 and loss=0.1025205237607536
epoch=3/10 and loss=0.09497572166506973
epoch=4/10 and loss=0.0885455057375095
epoch=5/10 and loss=0.084920114279627
epoch=6/10 and loss=0.07277637947341213
epoch=7/10 and loss=0.07469305278146235
epoch=8/10 and loss=0.06937187684930937
epoch=9/10 and loss=0.07345979950611678
epoch=10/10 and loss=0.05987445060189044


EVALUATION

In [19]:
correct_labels = 0
total_labels = 0
model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _,predicted=torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy={correct_labels/total_labels * 100}") 

accuracy=75.16000000000001
